# Reference sessions for the Python 3.12 environment migration

Runs the tongue-kinematics batch pipeline on 1–2 reference sessions and records the outputs, so
the Python 3.12 environment can be checked against the current 3.9 one. See
`PYTHON_311_UPGRADE_PLAN.md` in the library repo, Stage 2.

**Run it twice:**

1. **Now, in the original 3.9 capsule.** Leave `BASELINE_DIR = None`. Outputs go to
   `scratch/env_reference/py39/`. Save that folder as a data asset afterwards.
2. **Later, in the duplicate 3.12 capsule** (`env/py312` branch). Attach the py39 data asset,
   set `BASELINE_DIR` to the attached `py39` folder, and run all cells. It reruns *the same
   sessions* into `scratch/env_reference/py312/` and compares every output file to the baseline.

Nothing outside `scratch/env_reference/` is written. Code-Ocean-only, guarded by `IS_CO`.

## 1. Setup

In [1]:
import sys
import json
import platform
import subprocess
import datetime
from pathlib import Path

import numpy as np
import pandas as pd

if Path("/root/capsule").exists():
    SCRATCH = Path("/root/capsule/scratch")
    DATA = Path("/root/capsule/data")
    IS_CO = True
else:
    SCRATCH = None
    DATA = None
    IS_CO = False
    print("Local environment: nothing to run here. Run this notebook on Code Ocean.")

PY_TAG = "py{}{}".format(*sys.version_info[:2])
OUT_ROOT = SCRATCH / "env_reference" / PY_TAG if IS_CO else None
print("Python", platform.python_version(), "->", OUT_ROOT)

Python 3.9.12 -> /root/capsule/scratch/env_reference/py39


## 2. Parameters

- `BASELINE_DIR`: `None` on the 3.9 run. On the 3.12 run, set it to the attached py39 folder,
  the one that contains `reference_sessions.json`, for example
  `DATA / "<py39 asset name>" / "env_reference" / "py39"`. The sessions are then read from
  that file, so both runs use exactly the same ones.
- `REFERENCE_PRED_CSVS`: optionally pin the sessions yourself (a list of prediction CSV paths).
  Otherwise the first `N_SESSIONS` entries of `PRED_LIST_PATH` that exist on disk are used.

In [2]:
BASELINE_DIR = None

N_SESSIONS = 2
PRED_LIST_PATH = SCRATCH / "pred_csv_list_20250113.json" if IS_CO else None  # same list as run_batch_analysis.py
REFERENCE_PRED_CSVS = None

# Tolerance for numeric comparison (section 6). Values within it are reported as "close".
RTOL = 1e-6
ATOL = 1e-9

## 3. Choose sessions

In [3]:
if IS_CO:
    if BASELINE_DIR is not None:
        ref = json.loads((Path(BASELINE_DIR) / "reference_sessions.json").read_text())
        sessions = ref["pred_csvs"]
        print("Sessions from baseline (recorded on Python {}):".format(ref["python"]))
    elif REFERENCE_PRED_CSVS:
        sessions = [str(p) for p in REFERENCE_PRED_CSVS]
        print("Sessions pinned in REFERENCE_PRED_CSVS:")
    else:
        pred_list = json.loads(PRED_LIST_PATH.read_text())
        sessions = []
        for p in pred_list:
            if Path(p).exists():
                sessions.append(str(p))
            if len(sessions) == N_SESSIONS:
                break
        print("First {} existing entries of {}:".format(N_SESSIONS, PRED_LIST_PATH.name))

    missing = [p for p in sessions if not Path(p).exists()]
    assert sessions, "No sessions found. Check that data is attached, or set REFERENCE_PRED_CSVS."
    assert not missing, "Prediction CSVs not found (is the data attached?):\n  " + "\n  ".join(missing)
    for p in sessions:
        print("  ", p)

First 2 existing entries of pred_csv_list_20250113.json:
   /root/capsule/data/behavior_716325_2024-05-31_10-31-14_videoprocessed_2025-10-28_23-21-23/pred_outputs/video_preds/video_predictions.csv
   /root/capsule/data/behavior_717259_2024-06-28_11-17-19_videoprocessed_2025-10-28_23-21-23/pred_outputs/video_preds/video_predictions.csv


## 4. Run the pipeline

Same call as `run_batch_analysis.py`, but into `OUT_ROOT`, with `force_rerun=True` so every
session is recomputed. Clips are skipped. Expect a few minutes per session.

In [4]:
if IS_CO:
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_analysis import run_batch_analysis

    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    error_log = OUT_ROOT / "batch_error_log.txt"
    if error_log.exists():
        error_log.unlink()  # from an earlier run of this notebook

    t0 = datetime.datetime.now()
    run_batch_analysis(sessions, DATA, OUT_ROOT, extract_clips=False, force_rerun=True)
    print("Elapsed:", datetime.datetime.now() - t0)

    if error_log.exists():
        raise RuntimeError("Pipeline errors:\n" + error_log.read_text())


🔹 Starting analysis for: behavior_716325_2024-05-31_10-31-14

=== Generating tongue data for session: behavior_716325_2024-05-31_10-31-14 ===
Predictions CSV: /root/capsule/data/behavior_716325_2024-05-31_10-31-14_videoprocessed_2025-10-28_23-21-23/pred_outputs/video_preds/video_predictions.csv
keypoints extracted: ['nose_tip', 'jaw', 'tongue_tip_right', 'tongue_tip_center', 'tongue_tip_left', 'pointer_finger_r', 'paw_wrist_r', 'pointer_finger_l', 'paw_wrist_l', 'spout_r', 'spout_l']
Loaded keypoints: 11 raw dataframes
Found video CSV: /root/capsule/data/behavior_716325_2024-05-31_10-31-14/behavior-videos/BottomCamera/metadata.csv
Video QC: Frame numbers are sequential with no gaps.
Video QC: Timing differences are within expected range.
keypoint_df trimmed from 2689719 to 2689718
Synced keypoints
Segmented 7393 unique movements
Loading NWB from /root/capsule/data/foraging_nwb_bonsai/716325_2024-05-31_10-31-14.nwb
Timestamps are adjusted such that `_in_session` timestamps start at the

## 5. Record the environment

Writes `reference_sessions.json` (the sessions, Python and key package versions) and
`environment_freeze.txt` (`pip freeze`) next to the outputs.

In [5]:
if IS_CO:
    import aind_dynamic_foraging_behavior_video_analysis as lib
    import scipy
    import pynwb

    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True).stdout
    (OUT_ROOT / "environment_freeze.txt").write_text(freeze)
    lib_line = [l for l in freeze.splitlines() if "behavior-video-analysis" in l or "behavior_video_analysis" in l]

    record = {
        "pred_csvs": sessions,
        "python": platform.python_version(),
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "pynwb": pynwb.__version__,
        },
        "library": lib_line[0] if lib_line else lib.__file__,
    }
    (OUT_ROOT / "reference_sessions.json").write_text(json.dumps(record, indent=2))
    print(json.dumps(record, indent=2))

{
  "pred_csvs": [
    "/root/capsule/data/behavior_716325_2024-05-31_10-31-14_videoprocessed_2025-10-28_23-21-23/pred_outputs/video_preds/video_predictions.csv",
    "/root/capsule/data/behavior_717259_2024-06-28_11-17-19_videoprocessed_2025-10-28_23-21-23/pred_outputs/video_preds/video_predictions.csv"
  ],
  "python": "3.9.12",
  "created": "2026-09-24T20:48:00",
  "packages": {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scipy": "1.13.0",
    "pynwb": "3.0.0"
  },
  "library": "-e git+https://github.com/AllenNeuralDynamics/aind-dynamic-foraging-behavior-video-analysis.git@de27558a96a1104757082a565c6b688fd4d0dcff#egg=aind_dynamic_foraging_behavior_video_analysis"
}


## 6. Compare to the baseline (3.12 run only)

For each session, compares every `intermediate_data/*.parquet` file and
`tongue_quality_stats.json` with the baseline. Report statuses:

- **close**: numeric values differ, but within `RTOL`/`ATOL`. Expected from newer numpy/scipy.
- **note**: the dtype changed but the kind didn't, so values are still compared and match. The
  main expected case is pandas 3 storing timestamps in microseconds instead of nanoseconds
  (`datetime64[ns] -> datetime64[us]`). Harmless unless code depends on the unit.
- **DIFF**: anything else: missing files or columns, row-count changes, or values beyond
  tolerance. Look at each one before adopting the new environment.

Identical columns aren't listed. The report is also saved to `comparison_vs_baseline.csv`.

In [ ]:
def _is_num(s):
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)


def _is_text(s):
    return s.dtype == object or pd.api.types.is_string_dtype(s)


def _is_datetime(s):
    return pd.api.types.is_datetime64_any_dtype(s)


def _as_text(s):
    return s.astype(object).where(s.notna(), "<NA>").map(str)


def compare_frames(name, a, b):
    """Compare baseline frame a with new frame b; return report rows."""
    rows = []

    def add(check, status, detail=""):
        rows.append({"file": name, "check": check, "status": status, "detail": detail})

    only_a = [c for c in a.columns if c not in b.columns]
    only_b = [c for c in b.columns if c not in a.columns]
    if only_a or only_b:
        add("columns", "DIFF", "baseline only: {}; new only: {}".format(only_a, only_b))
    if len(a) != len(b):
        add("rows", "DIFF", "{} vs {}".format(len(a), len(b)))
        return rows
    if not a.index.equals(b.index):
        add("index", "DIFF", "index values differ")
    a = a.reset_index(drop=True)
    b = b.reset_index(drop=True)

    for c in [c for c in a.columns if c in b.columns]:
        x, y = a[c], b[c]
        if str(x.dtype) != str(y.dtype):
            same_kind = (_is_text(x) and _is_text(y)) or (_is_datetime(x) and _is_datetime(y))
            status = "note" if same_kind else "DIFF"
            add("dtype " + str(c), status, "{} -> {}".format(x.dtype, y.dtype))
        if _is_num(x) and _is_num(y):
            xv = x.to_numpy(dtype=float, na_value=np.nan)
            yv = y.to_numpy(dtype=float, na_value=np.nan)
            nan_mismatch = np.isnan(xv) != np.isnan(yv)
            if nan_mismatch.any():
                add("values " + str(c), "DIFF", "{} rows differ in NaN-ness".format(int(nan_mismatch.sum())))
                continue
            m = ~np.isnan(xv)
            if np.array_equal(xv[m], yv[m]):
                continue
            close = np.isclose(xv[m], yv[m], rtol=RTOL, atol=ATOL)
            add("values " + str(c), "close" if close.all() else "DIFF",
                "max abs diff {:.3g}; {} rows beyond tolerance".format(
                    np.abs(xv[m] - yv[m]).max(), int((~close).sum())))
        else:
            neq = _as_text(x) != _as_text(y)
            if neq.any():
                add("values " + str(c), "DIFF", "{} rows differ".format(int(neq.sum())))
    return rows


def _flatten(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(_flatten(v, "{}{}.".format(prefix, k)))
    elif isinstance(d, list):
        for i, v in enumerate(d):
            out.update(_flatten(v, "{}{}.".format(prefix, i)))
    else:
        out[prefix.rstrip(".")] = d
    return out


def _is_number(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)


def compare_json(name, a, b):
    """Compare two JSON documents key by key; return report rows."""
    fa, fb = _flatten(a), _flatten(b)
    rows = []
    for k in sorted(set(fa) | set(fb)):
        if k not in fa or k not in fb:
            rows.append({"file": name, "check": k, "status": "DIFF",
                         "detail": "missing in " + ("baseline" if k not in fa else "new")})
            continue
        va, vb = fa[k], fb[k]
        if _is_number(va) and _is_number(vb):
            if va == vb or (va != va and vb != vb):
                continue
            status = "close" if np.isclose(va, vb, rtol=RTOL, atol=ATOL) else "DIFF"
            rows.append({"file": name, "check": k, "status": status,
                         "detail": "{} -> {}".format(va, vb)})
        elif va != vb:
            rows.append({"file": name, "check": k, "status": "DIFF",
                         "detail": "{!r} -> {!r}".format(va, vb)})
    return rows


if IS_CO and BASELINE_DIR is None:
    print("BASELINE_DIR is None: this is the baseline run, so there's nothing to compare.")
    print("Next: save {} as a data asset (see section 7).".format(OUT_ROOT))

if IS_CO and BASELINE_DIR is not None:
    base = Path(BASELINE_DIR)
    rows = []
    session_dirs = sorted(d for d in base.iterdir() if d.is_dir())
    for sdir in session_dirs:
        new_sdir = OUT_ROOT / sdir.name
        base_files = sorted((sdir / "intermediate_data").glob("*.parquet"))
        for f in base_files:
            label = "{}/{}".format(sdir.name, f.name)
            new_f = new_sdir / "intermediate_data" / f.name
            if not new_f.exists():
                rows.append({"file": label, "check": "exists", "status": "DIFF", "detail": "missing in new run"})
                continue
            rows += compare_frames(label, pd.read_parquet(f), pd.read_parquet(new_f))
        stats = sdir / "tongue_quality_stats.json"
        if stats.exists():
            new_stats = new_sdir / stats.name
            label = "{}/{}".format(sdir.name, stats.name)
            if new_stats.exists():
                rows += compare_json(label, json.loads(stats.read_text()), json.loads(new_stats.read_text()))
            else:
                rows.append({"file": label, "check": "exists", "status": "DIFF", "detail": "missing in new run"})
        print("compared {}: {} parquet files + quality stats".format(sdir.name, len(base_files)))

    report = pd.DataFrame(rows, columns=["file", "check", "status", "detail"])
    report.to_csv(OUT_ROOT / "comparison_vs_baseline.csv", index=False)
    print()
    print(report["status"].value_counts().to_string() if len(report) else "Every column identical.")
    n_diff = int((report["status"] == "DIFF").sum()) if len(report) else 0
    print()
    if n_diff:
        print("[check] {} DIFF rows. Review them before adopting the new environment.".format(n_diff))
        display(report[report["status"] == "DIFF"])
    else:
        print("[ok] No DIFF rows: outputs match the baseline within tolerance.")

## 7. After running

- **3.9 run:** save `scratch/env_reference/py39/` as a Code Ocean data asset (for example
  "kinematics py39 reference outputs"). That asset is the comparison target, and it can't be
  modified.
- **3.12 run:** if there are no DIFF rows (or each one is understood), the environment is ready
  to adopt (plan step 2c). Save `scratch/env_reference/py312/`, including
  `comparison_vs_baseline.csv`, as a data asset too, as the record.